In [1]:
import numpy as np
import pandas as pd

from functools import partial
import importlib

import src.data
import src.features
import src.kernel_normalisation
import src.kernels
import src.krr
import src.metrics
import src.multikernel

from src.data import encode_labels, load_data, train_val_split
from src.features import HOG, HSVHistogram, ColorMoments, UniformLBPHistogram
from src.kernel_normalisation import unit_diag
from src.kernels import *
from src.krr import KernelRidgeRegression
from src.metrics import accuracy
from src.multikernel import *

In [2]:
DATA_DIR = "data/"

X_tr, X_te, y_tr_raw = load_data(DATA_DIR)

y_tr, inv_map = encode_labels(y_tr_raw)

n_classes = len(np.unique(y_tr))

### HOG features

In [3]:
hog = HOG(
    image_shape=(3, 32, 32),
    colour_mode="rgb",
    cell_size=8,
    block_size=2,
    block_stride=1,
    num_bins=6,
    signed=False,
).fit(X_tr)

X_tr_hog = hog.transform(X_tr)
X_te_hog = hog.transform(X_te)

X_train_hog, X_val_hog, y_train_hog, y_val_hog = train_val_split(
    X_tr_hog, y_tr, test_size=0.3, seed=20
)

### HSV Histogram features

In [5]:
hsv = HSVHistogram(bins=(8, 8, 8), grid=(4, 4)).fit(X_tr)

X_tr_hsv = hsv.transform(X_tr)
X_te_hsv = hsv.transform(X_te)

X_train_hsv, X_val_hsv, y_train_hsv, y_val_hsv = train_val_split(
    X_tr_hsv, y_tr, test_size=0.3, seed=20
)

### Color moment features

In [6]:
cm = ColorMoments(grid=(8, 8)).fit(X_tr)

X_tr_cm = cm.transform(X_tr)
X_te_cm = cm.transform(X_te)

X_train_cm, X_val_cm, y_train_cm, y_val_cm = train_val_split(
    X_tr_cm, y_tr, test_size=0.3, seed=20
)

### LBP features

In [7]:
lbp = UniformLBPHistogram(cell_size=8, pool="concat", per_channel=True).fit(X_tr)

X_tr_lbp = lbp.transform(X_tr)
X_te_lbp = lbp.transform(X_te)

X_train_lbp, X_val_lbp, y_train_lbp, y_val_lbp = train_val_split(
    X_tr_lbp, y_tr, test_size=0.3, seed=20
)

Grid search over kernel weights

In [10]:
gammas_hog = estimate_chi2_gammas_channel(X_train_hog)
hog_chi2 = partial(chi2_rbf_kernel_matrix_channel, gammas=gammas_hog)

gamma_hsv = estimate_gamma(X_train_hsv)
hsv_gaussian = partial(gaussian_kernel_matrix, gamma=gamma_hsv)

gamma_cm = estimate_laplacian_gamma(X_train_cm)
cm_laplacian = partial(laplacian_kernel_matrix, gamma=gamma_cm)

gamma_lbp = estimate_gamma(X_train_lbp)
lbp_gaussian = partial(gaussian_kernel_matrix, gamma=gamma_lbp)

# Order matters: (w_hog, w_hsv, w_cm, w_lbp)
specs4: list[KernelSpec] = [
    {
        "name": "hog_chi2",
        "Z": X_train_hog,
        "kernel_fn": hog_chi2,
        "diag_fn": unit_diag,
        "normalise": True,
    },
    {
        "name": "hsv_gaussian",
        "Z": X_train_hsv,
        "kernel_fn": hsv_gaussian,
        "diag_fn": unit_diag,
        "normalise": True,
    },
    {
        "name": "cm_laplacian",
        "Z": X_train_cm,
        "kernel_fn": cm_laplacian,
        "diag_fn": unit_diag,
        "normalise": True,
    },
    {
        "name": "lbp_gaussian",
        "Z": X_train_lbp,
        "kernel_fn": lbp_gaussian,
        "diag_fn": unit_diag,
        "normalise": True,
    },
]

# Local 3D grid around your prior good mixes:
# baseline guess ~ (hog=0.55, hsv=0.25, cm=0.20, lbp=0.00), but allow some lbp mass too.
# We grid (w_hog, w_hsv, w_cm); w_lbp = 1 - w_hog - w_hsv - w_cm.
w_hog_vals = np.array([0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65], dtype=np.float32)
w_hsv_vals = np.array([0.15, 0.20, 0.25, 0.30], dtype=np.float32)
w_cm_vals = np.array([0.10, 0.15, 0.20, 0.25], dtype=np.float32)

out4 = weight_grid_search_cv_four_kernels(
    specs4,
    y_train_hog,
    n_classes=n_classes,
    model="krr",
    w123_grid=(
        w_hog_vals.tolist(),
        w_hsv_vals.tolist(),
        w_cm_vals.tolist(),
    ),
    k=5,
    seed=20,
    lam=1e-4,
    verbose=1,
)

print(
    "best_w (hog, hsv, cm, lbp):",
    out4["best_w"],
    "best_mean_acc:",
    out4["best_mean_acc"],
)

w=(0.300,0.150,0.100,0.450) mean_acc=0.6191
w=(0.300,0.150,0.150,0.400) mean_acc=0.6249
w=(0.300,0.150,0.200,0.350) mean_acc=0.6263
w=(0.300,0.150,0.250,0.300) mean_acc=0.6237
w=(0.300,0.200,0.100,0.400) mean_acc=0.6234
w=(0.300,0.200,0.150,0.350) mean_acc=0.6254
w=(0.300,0.200,0.200,0.300) mean_acc=0.6263
w=(0.300,0.200,0.250,0.250) mean_acc=0.6229
w=(0.300,0.250,0.100,0.350) mean_acc=0.6240
w=(0.300,0.250,0.150,0.300) mean_acc=0.6240
w=(0.300,0.250,0.200,0.250) mean_acc=0.6237
w=(0.300,0.250,0.250,0.200) mean_acc=0.6234
w=(0.300,0.300,0.100,0.300) mean_acc=0.6263
w=(0.300,0.300,0.150,0.250) mean_acc=0.6254
w=(0.300,0.300,0.200,0.200) mean_acc=0.6237
w=(0.300,0.300,0.250,0.150) mean_acc=0.6214
w=(0.350,0.150,0.100,0.400) mean_acc=0.6214
w=(0.350,0.150,0.150,0.350) mean_acc=0.6214
w=(0.350,0.150,0.200,0.300) mean_acc=0.6234
w=(0.350,0.150,0.250,0.250) mean_acc=0.6226
w=(0.350,0.200,0.100,0.350) mean_acc=0.6211
w=(0.350,0.200,0.150,0.300) mean_acc=0.6254
w=(0.350,0.200,0.200,0.250) mean

## After tuning parameters, train the best model.

### Check the best model

In [11]:
w_hog, w_hsv, w_cm, w_lbp = np.asarray(out4["best_w"]).tolist()

# IMPORTANT: use the partial kernels you defined (with gammas)
Kh_tr = hog_chi2(X_train_hog, X_train_hog)
Ks_tr = hsv_gaussian(X_train_hsv, X_train_hsv)
Kc_tr = cm_laplacian(X_train_cm, X_train_cm)
Kl_tr = lbp_gaussian(X_train_lbp, X_train_lbp)

# Match CV: unit-diagonal normalisation per kernel
Kh_tr = normalise_train_gram(Kh_tr, unit_diag(X_train_hog))
Ks_tr = normalise_train_gram(Ks_tr, unit_diag(X_train_hsv))
Kc_tr = normalise_train_gram(Kc_tr, unit_diag(X_train_cm))
Kl_tr = normalise_train_gram(Kl_tr, unit_diag(X_train_lbp))

Ktr = w_hog * Kh_tr + w_hsv * Ks_tr + w_cm * Kc_tr + w_lbp * Kl_tr

# VAL x TRAIN cross Grams
Kh_val = hog_chi2(X_val_hog, X_train_hog)
Ks_val = hsv_gaussian(X_val_hsv, X_train_hsv)
Kc_val = cm_laplacian(X_val_cm, X_train_cm)
Kl_val = lbp_gaussian(X_val_lbp, X_train_lbp)

Kh_val = normalise_cross_gram(Kh_val, unit_diag(X_val_hog), unit_diag(X_train_hog))
Ks_val = normalise_cross_gram(Ks_val, unit_diag(X_val_hsv), unit_diag(X_train_hsv))
Kc_val = normalise_cross_gram(Kc_val, unit_diag(X_val_cm), unit_diag(X_train_cm))
Kl_val = normalise_cross_gram(Kl_val, unit_diag(X_val_lbp), unit_diag(X_train_lbp))

K_val = w_hog * Kh_val + w_hsv * Ks_val + w_cm * Kc_val + w_lbp * Kl_val

best_model = KernelRidgeRegression(n_classes=n_classes, kernel_fn=None, lam=1e-4).fit(
    None, y_train_hog, K=Ktr
)
yhat_val, _ = best_model.predict(K_star=K_val)
print("val acc:", accuracy(y_val_hog, yhat_val))

val acc: 0.6393333333333333


### Train on full dataset for submission

In [12]:
w_hog, w_hsv, w_cm, w_lbp = np.asarray(out4["best_w"]).tolist()

# Re-estimate gammas on FULL training set
gammas_hog = estimate_chi2_gammas_channel(X_tr_hog)
hog_chi2 = partial(chi2_rbf_kernel_matrix_channel, gammas=gammas_hog)

gamma_hsv = estimate_chi2_gamma(
    X_tr_hsv
)  # keep consistent with whatever you used above
hsv_gaussian = partial(gaussian_kernel_matrix, gamma=gamma_hsv)

gamma_cm = estimate_laplacian_gamma(X_tr_cm)
cm_laplacian = partial(laplacian_kernel_matrix, gamma=gamma_cm)

gamma_lbp = estimate_gamma(X_tr_lbp)
lbp_gaussian = partial(gaussian_kernel_matrix, gamma=gamma_lbp)

# FULL train Grams
Kh_tr = hog_chi2(X_tr_hog, X_tr_hog)
Ks_tr = hsv_gaussian(X_tr_hsv, X_tr_hsv)
Kc_tr = cm_laplacian(X_tr_cm, X_tr_cm)
Kl_tr = lbp_gaussian(X_tr_lbp, X_tr_lbp)

Kh_tr = normalise_train_gram(Kh_tr, unit_diag(X_tr_hog))
Ks_tr = normalise_train_gram(Ks_tr, unit_diag(X_tr_hsv))
Kc_tr = normalise_train_gram(Kc_tr, unit_diag(X_tr_cm))
Kl_tr = normalise_train_gram(Kl_tr, unit_diag(X_tr_lbp))

Ktr = w_hog * Kh_tr + w_hsv * Ks_tr + w_cm * Kc_tr + w_lbp * Kl_tr

best_model = KernelRidgeRegression(n_classes=n_classes, kernel_fn=None, lam=1e-4).fit(
    None, y_tr, K=Ktr
)

# TEST x TRAIN cross Grams
Kh_te_tr = hog_chi2(X_te_hog, X_tr_hog)
Ks_te_tr = hsv_gaussian(X_te_hsv, X_tr_hsv)
Kc_te_tr = cm_laplacian(X_te_cm, X_tr_cm)
Kl_te_tr = lbp_gaussian(X_te_lbp, X_tr_lbp)

Kh_te_tr = normalise_cross_gram(Kh_te_tr, unit_diag(X_te_hog), unit_diag(X_tr_hog))
Ks_te_tr = normalise_cross_gram(Ks_te_tr, unit_diag(X_te_hsv), unit_diag(X_tr_hsv))
Kc_te_tr = normalise_cross_gram(Kc_te_tr, unit_diag(X_te_cm), unit_diag(X_tr_cm))
Kl_te_tr = normalise_cross_gram(Kl_te_tr, unit_diag(X_te_lbp), unit_diag(X_tr_lbp))

K_star = w_hog * Kh_te_tr + w_hsv * Ks_te_tr + w_cm * Kc_te_tr + w_lbp * Kl_te_tr

yte_int, _ = best_model.predict(K_star=K_star)
yte = np.array([inv_map[i] for i in yte_int])

sub = pd.DataFrame({"Prediction": yte})
sub.index += 1
sub.to_csv("results/submission.csv", index_label="Id")